# Week 1: Python for Security
**Goal:** use basic Python to investigate a server log and catch a brute-force attack.

**Time:** about 2 hours. **You need:** basic Python (strings, lists, loops, dictionaries).

**Story.** You are the new junior analyst at a small company. Last night the server `lab-server` was slow.
Your manager hands you its SSH login log (`auth.log`) and asks: *Did someone try to break in? Did they get in?*

> **Rules:** this log is synthetic. All IP addresses are from documentation ranges and point at no real computer.
> Only analyze data the lab gives you. You signed the ethics agreement, right?

## Part 0: Get the data (run this cell)

In [ ]:
import os, urllib.request

DATA_URL = "https://raw.githubusercontent.com/Pantea1358/Cyber-labs-/main/weeks/week01-python-for-security/data/"
os.makedirs("data", exist_ok=True)
for name in ["auth.log"]:
    if not os.path.exists(f"data/{name}"):
        urllib.request.urlretrieve(DATA_URL + name, f"data/{name}")
print("Data ready:", os.listdir("data"))

## Part 1: Read the log (10 min)
A log file is just text: one event per line. Read the file into a list called `lines`, one string per line.

Then print how many lines there are and the first 5 lines.

*Hint:* `open(...).read().splitlines()`

In [ ]:
# TODO: read data/auth.log into a list called `lines`
lines = []

print("Number of lines:", len(lines))
# TODO: print the first 5 lines

**Look at the lines.** Each line has a date and time, the server name, the program (`sshd`, the SSH server),
and a message. The messages you care about are:

- `Accepted password for USER from IP ...` : a successful login
- `Failed password for USER from IP ...` : a wrong password
- `Failed password for invalid user USER from IP ...` : a wrong password for a user that does not even exist

**Question 1 (write in this cell):** why would someone try to log in as a user that does not exist?

*Your answer:*

## Part 2: Find the failed logins (10 min)
Make a list `failed` with only the lines that contain `Failed password`. Print how many there are.

In [ ]:
# TODO: keep only lines that contain "Failed password"
failed = []
print("Failed logins:", len(failed))

## Part 3: Parse each line into fields (20 min)
Counting lines is not enough. We need to know **who** and **from where**.

Write a function `parse_line(line)` that returns a dictionary like:

```python
{"time": "Nov 03 10:58:00", "user": "bob", "ip": "10.0.0.15", "status": "failed"}
```

`status` should be `"failed"`, `"accepted"`, or `"other"`. Only login attempts matter here, and
they all contain the words `password for`. Return `None` for every other line (like `Disconnected from user ...`).

*Hint:* `words = line.split()`. The IP is always the word right **after** `"from"`, and the user is the word right **before** `"from"`.
Use `words.index("from")` instead of a fixed position. You will see why in Part 6.

In [ ]:
def parse_line(line):
    # TODO: return None if the line does not contain "password for"
    words = line.split()
    # TODO: find the position of "from", then the user and IP around it
    # TODO: work out status: "failed", "accepted", or "other"
    return None

events = [e for e in (parse_line(l) for l in lines) if e is not None]
print(len(events), "events parsed")
print(events[:3])

**Check your work.** Run this cell. If you see no error, your parser works.

In [ ]:
test1 = "Nov 03 10:58:00 lab-server sshd[2004]: Failed password for bob from 10.0.0.15 port 53880 ssh2"
test2 = "Nov 03 18:02:11 lab-server sshd[2101]: Failed password for invalid user admin from 203.0.113.45 port 41000 ssh2"
test3 = "Nov 03 09:40:00 lab-server sshd[2002]: Disconnected from user alice 10.0.0.12 port 51285"
assert parse_line(test1) == {"time": "Nov 03 10:58:00", "user": "bob", "ip": "10.0.0.15", "status": "failed"}
assert parse_line(test2)["user"] == "admin" and parse_line(test2)["ip"] == "203.0.113.45"
assert parse_line(test3) is None
print("All checks passed!")

## Part 4: Count failures per IP address (10 min)
Build a dictionary `fails_per_ip` that maps each IP to its number of **failed** logins.
Then print the top 5 IPs, most failures first.

*Hint:* `d[key] = d.get(key, 0) + 1`, and `sorted(d.items(), key=lambda kv: kv[1], reverse=True)`.

In [ ]:
fails_per_ip = {}
# TODO: loop over events and count failed logins per IP

# TODO: sort and print the top 5

## Part 5: Flag suspicious IPs (5 min)
A person who mistypes a password fails once or twice. A program guessing passwords fails many times.

Make a list `suspicious` of IPs with **10 or more** failures.

**Question 2:** do any suspicious IPs start with `10.`? Those are inside our company network. What would that mean?

*Your answer:*

In [ ]:
THRESHOLD = 10
suspicious = []  # TODO
print("Suspicious IPs:", suspicious)

## Part 6: Did anyone get in? (10 min)
This is the most important question. Find every suspicious IP that **also** has an `accepted` login.
Print the IP, the user, and the time. That account is probably compromised.

In [ ]:
# TODO: print an ALERT for every accepted login from a suspicious IP

**Question 3:** you found a compromised account. List three things the company should do next.

*Your answer:*

## Part 7: AI twist, can you trust AI-written code? (30 min)
A classmate asked an AI assistant to "write a Python function that gets the IP address from an SSH log line".
This is what it gave them:

In [ ]:
# Written by an AI assistant
def get_ip(line):
    return line.split()[10]

print(get_ip(test1))
print(get_ip(test2))

1. Run the cell above. Is the output right for **both** test lines?
2. Explain in your own words why it fails on `test2`.
3. Paste the function into an AI assistant (if your school allows it) and ask: *"Is there a bug in this function for SSH auth log lines?"*
   Does it find the bug? Does its fix pass the tests below?
4. Write your own fixed `get_ip_fixed` and make the checks pass.

Write your answers to 1 to 3 here:

*Your answers:*

In [ ]:
def get_ip_fixed(line):
    # TODO: fix the bug
    return None

assert get_ip_fixed(test1) == "10.0.0.15"
assert get_ip_fixed(test2) == "203.0.113.45"
wrong = sum(1 for l in failed if get_ip(l) != get_ip_fixed(l))
print(f"The AI version is wrong on {wrong} of {len(failed)} failed-login lines")

**Lesson:** AI code often works on the first example and breaks on real data.
In security, a parser that silently returns the wrong IP means you block the wrong address, or miss the attacker.
Always test AI code on tricky inputs.

## Part 8: Reflect (15 min)
**Exit ticket.** Answer in 2 to 3 sentences each:

1. You ran the same analysis on a log from a website you do not own. Is that okay? Why or why not?
2. Name one way AI helped or could help a security analyst, and one way it could mislead them.

*Your answers:*

**Stretch (optional):** add a check for *time*. Which suspicious IP made the most attempts per minute?